# XD2-Net — 02: Training

Trains one model per prefix-length bucket using the artifacts saved by
`01_data_preparation.ipynb`. Saves the best checkpoint plus a `model_config_<bucket>.json`
per bucket, and plots loss curves.

**Prerequisite:** run notebook 01 first with the same `MODEL` / `DATASET_CONFIG`.

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/xd2-net'
    %pip install -e {REPO_ROOT} --quiet
else:
    REPO_ROOT = os.path.abspath('..')
    sys.path.insert(0, REPO_ROOT)

os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)

In [ ]:
# --- parameters --------------------------------------------------------------
MODEL = 'baseline'                                   # 'baseline' | 'hierarchical'
DATASET_CONFIG = 'configs/datasets/bpic2012.yaml'    # any dataset YAML (see configs/datasets/_template.yaml)
MODEL_CONFIG = None                                  # None = configs/models/<MODEL>.yaml

In [ ]:
import glob
import json
import logging
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import torch

from xd2net.common.config import load_dataset_config, load_experiment_config
from xd2net.common.data import make_dataloaders
from xd2net.common.registry import get_spec

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s',
                    datefmt='%H:%M:%S')

spec = get_spec(MODEL)
exp = load_experiment_config(MODEL_CONFIG or spec.default_config, spec.model_config_cls)
ds = load_dataset_config(DATASET_CONFIG)
run_dir = os.path.join(exp.workspace_dir, MODEL, ds.name)
artifacts_dir = os.path.join(run_dir, 'artifacts')
ckpt_dir = os.path.join(run_dir, 'checkpoints')

def available_buckets():
    """Bucket keys with a saved artifact, in prefix-length order."""
    keys = [os.path.basename(p)[len('bucket_'):-len('.pkl')]
            for p in glob.glob(os.path.join(artifacts_dir, 'bucket_*.pkl'))]
    return sorted(keys, key=lambda k: int(k.split('_')[1]))

print('Artifacts dir:', artifacts_dir)

In [ ]:
# ── Optional: train only a subset of buckets ─────────────────────────────────
# Set to None to train all buckets, or e.g. ['1_10', '1_15'] for a quick test
BUCKETS_TO_TRAIN = None  # e.g. ['1_10', '1_15']

In [ ]:
results_rows = []
histories = {}

for bucket_key in available_buckets():
    if BUCKETS_TO_TRAIN and bucket_key not in BUCKETS_TO_TRAIN:
        continue

    with open(os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl'), 'rb') as f:
        art = pickle.load(f)

    print(f'\n{"="*60}\nBucket {bucket_key}')

    torch.manual_seed(exp.training.seed)
    model = spec.build_model(art, exp.model, ds.name)
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

    train_loader, val_loader = make_dataloaders(
        art['inputs_train'], art['outputs_train'], art['lengths_train'],
        val_split=exp.training.val_split,
        batch_size=exp.training.batch_size,
        seed=exp.training.seed,
    )

    trainer = spec.trainer_cls(model, exp.training, ckpt_dir, bucket=bucket_key)
    history = trainer.train(train_loader, val_loader)
    histories[bucket_key] = history
    model = trainer.load_best_model()
    os.makedirs(ckpt_dir, exist_ok=True)
    with open(os.path.join(ckpt_dir, f'model_config_{bucket_key}.json'), 'w') as f:
        json.dump(model.get_config(), f, indent=2)

    results_rows.append({'bucket': bucket_key, 'best_val_loss': min(history['val_loss']),
                         'best_val_auc': max(history['val_auc']), 'epochs': len(history['val_loss'])})

print('\nTraining complete.')

In [ ]:
# ── Plot loss curves ──────────────────────────────────────────────────────────
for bucket_key, h in histories.items():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(h['train_loss'], label='train')
    ax1.plot(h['val_loss'],   label='val')
    ax1.set_title(f'{bucket_key} — Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss'); ax1.legend()

    ax2.plot(h['val_auc'])
    ax2.set_title(f'{bucket_key} — Val AUC')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('ROC-AUC')

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
df_summary = pd.DataFrame(results_rows)
print(df_summary.to_string(index=False))